# AstroSpace: a size-controlled, exhaustively enumerated CHNO chemical space

This notebook constructs AstroSpace, an exhaustively enumerated chemical space of neutral, closed-shell
molecules of carbon, hydrogen, nitrogen and oxygen with up to six heavy atoms, computes the dipole moment,
rotational constants and energy of every molecule with the semiempirical GFN2-xTB method, evaluates
predictors of interstellar detection status under a same-size protocol, and reproduces every figure,
table and number of the accompanying manuscript from scratch. No pre-computed data or trained models are
required.

## Overview

Machine-learning studies that prioritize molecules for interstellar detection typically draw candidates
from the QM9 dataset. QM9 is dominated by nine-heavy-atom molecules, whereas confirmed interstellar
molecules are small, so a ranker that uses heavy-atom count alone nearly saturates the conventional
whole-dataset AUC. AstroSpace removes this confound: every confirmed interstellar molecule in range is
compared only with undetected molecules of the same heavy-atom count (the same-size AUC), so that
molecular size cannot contribute to the score.

## Pipeline

| Module | Description |
|---|---|
| **M1** | Exhaustive enumeration of neutral closed-shell CHNO molecules (≤ 6 heavy atoms) |
| **M2** | GFN2-xTB geometry, dipole moment, rotational constants and energy for each molecule |
| **M3** | Δ-ML dipole correction toward the B3LYP/6-31G(2df,p) values of QM9, with Mondrian conformal intervals (sensitivity analysis) |
| **M4** | Same-size evaluation of six predictors with bootstrap confidence intervals and permutation tests |
| **M4b** | Counterfactual size experiment (Table 1) and a graph isomorphism network under the identical protocol |
| **M5** | Ranked candidate list after a strain screen and a dipole-moment filter (Tables S1 and S2), and the prospective check of the molecules reported after the census cut-off (Table S3) |
| **M6** | Figures 1–5 (300 dpi PNG and PDF) |
| **M7** | Figures S1–S5 and the graphical abstract |

## Requirements

The pipeline runs on Google Colab or Kaggle with a CPU runtime; no GPU is used. The only external
input, required solely for the M3 sensitivity analysis, is `QM9_dipole_reference.csv` (released with
the code). On Kaggle, use **Accelerator = None** (a GPU or TPU session is not used and, because of CPU
oversubscription, can be much slower), add the reference file as a dataset input and enable Internet
access so that `pip` can install `tblite`. The GFN2-xTB step is CPU-bound and checkpoints every 500 molecules; with one single-threaded
worker per core it takes roughly 0.15 core-seconds per molecule, i.e. about 20–30 minutes for the whole
space on the four cores of a standard Kaggle CPU session. The graph-network training (M4b, three
networks per fold) takes about 15–20 minutes on four cores; all other modules together a few minutes.

## Scope and default constraints

The chemical space is restricted to neutral, closed-shell CHNO molecules. Two constraints are applied
by default and are justified in the manuscript on chemical grounds (Section 3.1.2), independently of
the results; each can be relaxed through a flag in the configuration cell to reproduce the
corresponding control experiment (Figure S2).

- **Formally charge-separated valence structures are excluded** (`ALLOW_CHARGE_SEPARATION = False`).
  Such structures are alternative representations of neutral molecules rather than distinct species;
  admitting them enlarges the space with ylide-like graphs. Five detected molecules that can only be
  written with formal charges (CO, HNC, N2O, CH3NC, HCNO) are therefore absent by construction.
- **Fluorine is excluded** (`INCLUDE_FLUORINE = False`). No confirmed interstellar molecule in this size
  range contains fluorine, and its valence of one multiplies the space combinatorially.

Ranking uses the raw GFN2-xTB dipole moment throughout; the Δ-ML-corrected values are reported as a
sensitivity analysis only (Table 2 and Figure S2c of the manuscript).

## Reproducibility

The enumeration is deterministic and its output is sorted, the three-dimensional embedding uses a fixed
random seed, and every random sample, bootstrap and permutation test is seeded. The number of converged
GFN2-xTB structures can differ by a few molecules between computing environments, because the seeded
embedding is not bit-for-bit reproducible across RDKit builds; the same-size AUC values of the trained
predictors are then reproduced to within about ±0.01, well inside their bootstrap intervals. Every
number quoted in the manuscript is written to `paper_numbers.json`.


In [ ]:
# ============================================================
# CELL 0 — Preflight: platform, working directory, reference file
# ============================================================
import sys, os, glob
from pathlib import Path
# The GFN2-xTB step (M2) runs in a process pool with one molecule per task, so every process must stay
# single-threaded: otherwise the OpenMP threads of tblite and BLAS oversubscribe the cores. On Kaggle
# GPU/TPU sessions the host reports far more CPUs than the container may use, which makes the
# oversubscription catastrophic (hours instead of minutes). Set before any numerical library is imported.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"
N_CORES = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)

ON_KAGGLE = Path('/kaggle/working').exists()
ON_COLAB  = Path('/content').exists() and not ON_KAGGLE
PLATFORM  = "Kaggle" if ON_KAGGLE else ("Colab" if ON_COLAB else "local")
BASE      = Path('/kaggle/working') if ON_KAGGLE else (Path('/content') if ON_COLAB else Path.cwd())

print(f"platform : {PLATFORM}")
print(f"workdir  : {BASE}")
print(f"CPU cores usable by this session: {N_CORES}  (the GFN2-xTB step is CPU-bound; no GPU is used)")
if ON_KAGGLE:
    print("Kaggle   : Accelerator = None (a GPU or TPU session is not used and can be much slower);")
    print("           Settings -> Internet must be ON (pip installs tblite); add QM9_dipole_reference.csv")
    print("           as a dataset input (it is found at any depth under /kaggle/input/).")

def discover(name):
    """Locate an input file next to the notebook, in ./data, or anywhere under /kaggle/input."""
    roots = [BASE, BASE/'data']
    if ON_KAGGLE: roots += [Path('/kaggle/input')]
    for r in roots:
        p = r/name
        if p.exists(): return p
    for r in roots:
        hits = glob.glob(str(r/'**'/name), recursive=True)
        if hits: return Path(sorted(hits)[0])
    return None

QM9_REF = discover('QM9_dipole_reference.csv')          # the only external input (M3 sensitivity analysis)
print("\nQM9 reference:", QM9_REF if QM9_REF else "NOT FOUND (M3 is skipped; raw GFN2-xTB dipole moments are used)")


In [ ]:
# ============================================================
# CELL 1 — Install (tblite provides the GFN2-xTB engine)
# ============================================================
!pip install -q rdkit ase tblite xgboost scikit-learn pandas matplotlib scipy

missing=[]
for mod in ["rdkit","ase","tblite","xgboost","sklearn","scipy"]:
    try: __import__(mod)
    except ImportError: missing.append(mod)
if missing:
    raise SystemExit(f"Install failed for {missing}. On Kaggle enable Internet (Settings) and rerun this cell.")
from tblite.interface import Calculator
print("Packages ready. GFN2-xTB available through tblite.")

In [ ]:
# ============================================================
# CELL 2 — Config
# ============================================================
import os, math, json, time, pickle, warnings, contextlib, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*'); warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

MAX_HEAVY   = 6      # heavy-atom ceiling of the enumeration: 6 -> 35,482 molecules | 5 -> 4,141 | 4 -> 571
# Formally charge-separated valence structures are excluded by default (manuscript Section 3.1.2).
# Turning the flag ON admits CO/HNC/N2O/CH3NC/HCNO together with thousands of ylide-like graphs whose
# GFN2-xTB dipole moments are overestimated (control experiment, Figure S2a).
ALLOW_CHARGE_SEPARATION = False

# Fluorine is excluded by default (CHNO, not CHNOF): no confirmed interstellar molecule in this size
# range contains fluorine. With the flag ON, about a quarter of the space and half of the candidate
# list are fluorinated (control experiment, Figure S2b).
INCLUDE_FLUORINE = False

# Ranking uses the raw GFN2-xTB dipole moment; the Delta-ML-corrected values (M3) are a sensitivity
# analysis reported in Table 2 and Figure S2c only.
RANK_DIPOLE = "mu_xtb"
N_PROC      = max(1, N_CORES)                  # one single-threaded xTB process per usable core; a GPU does not help
XTB_STEPS   = 40     # maximum BFGS steps per geometry optimisation

WONG = dict(black="#000000", orange="#E69F00", skyblue="#56B4E9", green="#009E73",
            yellow="#F0E442", blue="#0072B2", vermillion="#D55E00", purple="#CC79A7")
mpl.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"savefig.bbox":"tight",
  "font.family":"DejaVu Sans","axes.spines.top":False,"axes.spines.right":False,
  "axes.grid":True,"grid.alpha":0.25,"font.size":11})

OUT = BASE/"astrospace"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT/"figures"; FIG.mkdir(exist_ok=True)
CKPT = OUT/"xtb_checkpoint.csv"
print(f"MAX_HEAVY={MAX_HEAVY} | processes={N_PROC} | outputs -> {OUT}")
# about 0.15 core-seconds per molecule with single-threaded workers (see the preflight cell)
if ALLOW_CHARGE_SEPARATION: _n = {4:2361,5:22841,6:200000}.get(MAX_HEAVY,22841)
elif INCLUDE_FLUORINE:       _n = {4:737,5:5452,6:47517}.get(MAX_HEAVY,5452)
else:                        _n = {4:571,5:4141,6:35482}.get(MAX_HEAVY,35482)   # CHNO
_est = _n * 0.15 / N_PROC / 60
print(f"estimated xTB wall-clock: ~{_est:.0f} min on {N_PROC} core(s). Checkpointed every 500 molecules.")

---
## M1 — Exhaustive enumeration

Every neutral, closed-shell molecule of carbon (valence 4), nitrogen (3) and oxygen (2) with at most
`MAX_HEAVY` heavy atoms is enumerated; hydrogens fill the remaining valence implicitly. Molecules are
grown one heavy atom at a time, ring closures and bond-order increases are applied to a fixed point at
each level, and duplicates are removed by canonical SMILES. The enumeration is performed at the level of
constitutional isomers (no stereochemistry) and involves no random choices; its output is sorted so that
every downstream step sees the molecules in the same order on every platform. Completeness is checked
against the confirmed interstellar molecules in range and against the neutral CHNO molecules of the QM9
reference file.


In [ ]:
# ============================================================
# CELL 3 — M1: exhaustive enumeration of CHNOF molecules
# ============================================================
# Element set is controlled by INCLUDE_FLUORINE (defined in the config cell above).
NEUTRAL = {(6,0):4, (7,0):3, (8,0):2}                 # C, N, O
if INCLUDE_FLUORINE: NEUTRAL[(9,0)] = 1                # add F (valence 1) only if explicitly enabled
YLIDE   = {(6,-1):3, (7,1):4, (8,1):3, (7,-1):2, (8,-1):1}
VALS    = {**NEUTRAL, **YLIDE} if ALLOW_CHARGE_SEPARATION else NEUTRAL
BT = {1:Chem.BondType.SINGLE, 2:Chem.BondType.DOUBLE, 3:Chem.BondType.TRIPLE}

def canon(rw):
    """Canonical SMILES. Bracket atoms freeze their H count, so clear it and let
    RDKit recompute implicit hydrogens; then require a clean SMILES round-trip."""
    try:
        r = Chem.RWMol(rw)
        for a in r.GetAtoms():
            a.SetNumExplicitHs(0); a.SetNoImplicit(False)
        mm = Chem.Mol(r); Chem.SanitizeMol(mm)
        s = Chem.MolToSmiles(mm)
        return s if Chem.MolFromSmiles(s) is not None else None
    except Exception:
        return None

def net_charge(m):  return sum(a.GetFormalCharge() for a in m.GetAtoms())
def charged(m):     return [a.GetIdx() for a in m.GetAtoms() if a.GetFormalCharge()!=0]

def adjacency_ok(m):
    """Formal charges only as a BONDED +/- pair (a dative / ylide bond).
    This admits CO = [C-]#[O+], HNC, N2O, CH3NC, HCNO, while excluding
    zwitterions such as [NH3+]CC(=O)[O-], which are not stable in the gas phase."""
    ci = charged(m)
    if len(ci) < 2: return True
    if len(ci) > 2: return False
    i,j = ci
    if m.GetAtomWithIdx(i).GetFormalCharge() + m.GetAtomWithIdx(j).GetFormalCharge() != 0: return False
    return m.GetBondBetweenAtoms(i,j) is not None

def free_val(m, i):
    a = m.GetAtomWithIdx(i); k = (a.GetAtomicNum(), a.GetFormalCharge())
    if k not in VALS: return -99
    return VALS[k] - sum(int(b.GetBondTypeAsDouble()) for b in a.GetBonds())

def viable(m):
    if abs(net_charge(m)) > 1 or len(charged(m)) > 2: return False      # charged intermediates allowed
    if ALLOW_CHARGE_SEPARATION and net_charge(m)==0 and len(charged(m))==2 and not adjacency_ok(m):
        return False
    return True

def _atom(Z,c):
    a = Chem.Atom(Z); a.SetFormalCharge(c); return a

def grow_atom(m):
    out=[]
    for i in range(m.GetNumAtoms()):
        f = free_val(m,i)
        if f<=0: continue
        for (Z,c) in VALS:
            for order in range(1, min(f, VALS[(Z,c)], 3)+1):
                rw = Chem.RWMol(m); j = rw.AddAtom(_atom(Z,c)); rw.AddBond(i,j,BT[order])
                if not viable(rw): continue
                s = canon(rw)
                if s: out.append(s)
    return out

def close_or_raise(m):
    out=[]; n=m.GetNumAtoms()
    for i in range(n):
        for j in range(i+1,n):
            fi,fj = free_val(m,i), free_val(m,j)
            if fi<=0 or fj<=0: continue
            b = m.GetBondBetweenAtoms(i,j); lim = min(fi,fj,3)
            cur = 0 if b is None else int(b.GetBondTypeAsDouble())
            orders = range(1,lim+1) if b is None else range(cur+1, min(cur+lim,3)+1)
            for o in orders:
                rw = Chem.RWMol(m)
                if b is None: rw.AddBond(i,j,BT[o])
                else:         rw.GetBondBetweenAtoms(i,j).SetBondType(BT[o])
                if not viable(rw): continue
                s = canon(rw)
                if s: out.append(s)
    return out

def saturate(S):
    seen=set(S); frontier=set(S)
    while frontier:
        nxt=set()
        for s in frontier:
            m=Chem.MolFromSmiles(s)
            if m is None: continue
            for c in close_or_raise(m):
                if c not in seen: seen.add(c); nxt.add(c)
        frontier=nxt
    return seen

t0=time.time(); levels={}
seeds=set()
for (Z,c) in VALS:
    rw=Chem.RWMol(); rw.AddAtom(_atom(Z,c))
    s=canon(rw)
    if s: seeds.add(s)
levels[1]=saturate(seeds)
for n in range(2, MAX_HEAVY+1):
    nxt=set()
    for s in levels[n-1]:
        m=Chem.MolFromSmiles(s)
        if m is None: continue
        for c in grow_atom(m): nxt.add(c)
    nxt = saturate(nxt)
    keep=set()
    for s in nxt:
        m=Chem.MolFromSmiles(s)
        if m is not None and m.GetNumHeavyAtoms()==n: keep.add(s)
    levels[n]=keep
    print(f"  n={n}: {len(keep):>7,} raw   [{time.time()-t0:.0f}s]")

# keep only net-neutral molecules obeying the charge rule
final={}
for n,S in levels.items():
    k=set()
    for s in S:
        m=Chem.MolFromSmiles(s)
        if m is None or net_charge(m)!=0: continue
        if ALLOW_CHARGE_SEPARATION and not adjacency_ok(m): continue
        k.add(s)
    final[n]=k

space = pd.DataFrame([(s,n) for n,S in final.items() for s in S],
                     columns=["smiles","heavy_atoms"]).drop_duplicates("smiles")
# The enumeration uses Python sets, whose iteration order differs between processes; sorting makes the
# row order, and hence every seeded random sample downstream, identical on every platform.
space = space.sort_values(["heavy_atoms","smiles"], kind="mergesort").reset_index(drop=True)
n_cs = sum(len(charged(Chem.MolFromSmiles(s)))>0 for s in space.smiles)
print(f"\nAstroSpace-{MAX_HEAVY}: {len(space):,} molecules in {time.time()-t0:.0f}s")
print(f"  of which charge-separated (CO-like dative bonds): {n_cs:,}")


In [ ]:
# ============================================================
# CELL 4 — M1b: interstellar labels + validation assertions
# ============================================================
# Confirmed interstellar and circumstellar molecules (McGuire, ApJS 2022, 259, 30, "2021 census", as maintained in the
# astromol package, v2021.7.0) that are neutral, closed-shell, composed of C, H, N and O, and have at most six heavy atoms.
# Values are (SMILES, year of first report). Only molecules reported through 2021 are positives.
ISM = {
    "methane": ('C', 1991),
    "water": ('O', 1969),
    "ammonia": ('N', 1968),
    "formaldehyde": ('C=O', 1969),
    "methanol": ('CO', 1970),
    "hydrogen cyanide": ('C#N', 1971),
    "methanimine": ('C=N', 1973),
    "methylamine": ('CN', 1974),
    "acetylene": ('C#C', 1976),
    "nitroxyl HNO": ('N=O', 1977),
    "ethylene": ('C=C', 1981),
    "molecular nitrogen": ('N#N', 2004), "molecular oxygen": ('O=O', 2007),
    "hydrogen peroxide": ('OO', 2011),
    "hydroxylamine": ('NO', 2020),
    "formic acid": ('OC=O', 1971),
    "acetonitrile": ('CC#N', 1971),
    "formamide": ('NC=O', 1971),
    "isocyanic acid HNCO": ('N=C=O', 1972),
    "acetaldehyde": ('CC=O', 1973),
    "propyne": ('CC#C', 1973),
    "dimethyl ether": ('COC', 1974),
    "cyanamide": ('NC#N', 1975),
    "ethanol": ('CCO', 1975),
    "ketene": ('C=C=O', 1977),
    "carbon dioxide": ('O=C=O', 1989),
    "ethylene oxide": ('C1CO1', 1997),
    "vinyl alcohol": ('C=CO', 2001),
    "ketenimine": ('C=C=N', 2006),
    "propene": ('CC=C', 2007),
    "cyanic acid HOCN": ('OC#N', 2009),
    "carbodiimide": ('N=C=N', 2012),
    "ethanimine": ('CC=N', 2013),
    "nitrous acid HONO": ('O=NO', 2019),
    "vinylamine": ('C=CN', 2021),
    "cyanoacetylene": ('C#CC#N', 1971),
    "vinyl cyanide": ('C=CC#N', 1975),
    "methyl formate": ('COC=O', 1975),
    "ethyl cyanide": ('CCC#N', 1977),
    "acetone": ('CC(C)=O', 1987),
    "propynal": ('C#CC=O', 1988),
    "acetic acid": ('CC(=O)O', 1997),
    "glycolaldehyde": ('OCC=O', 2000),
    "diacetylene": ('C#CC#C', 2001),
    "ethylene glycol": ('OCCO', 2002),
    "propenal": ('C=CC=O', 2004),
    "propanal": ('CCC=O', 2004),
    "cyclopropenone": ('O=C1C=C1', 2006),
    "acetamide": ('CC(N)=O', 2006),
    "cyanoformaldehyde": ('N#CC=O', 2007),
    "aminoacetonitrile": ('NCC#N', 2008),
    "E-cyanomethanimine": ('N=CC#N', 2013),
    "methyl isocyanate": ('CN=C=O', 2015),
    "propylene oxide": ('CC1CO1', 2016),
    "methoxymethanol": ('COCO', 2017),
    "ethyl methyl ether": ('CCOC', 2018),
    "glycolonitrile": ('OCC#N', 2019),
    "urea": ('NC(N)=O', 2019),
    "propargylimine": ('C#CC=N', 2020),
    "vinylacetylene": ('C=CC#C', 2021),
    "N-methylformamide": ('CNC=O', 2021),
    "ethanolamine": ('NCCO', 2021),
    "methyl cyanoacetylene": ('CC#CC#N', 1984),
    "methyldiacetylene": ('CC#CC#C', 1984),
    "cyanoallene": ('C=C=CC#N', 2006),
    "ethyl formate": ('CCOC=O', 2009),
    "n-propyl cyanide": ('CCCC#N', 2009),
    "methyl acetate": ('CC(=O)OC', 2013),
    "isopropyl cyanide": ('CC(C)C#N', 2014),
    "propargyl cyanide": ('C#CCC#N', 2020),
    "hydroxyacetone": ('CC(=O)CO', 2020),
    "allenyl acetylene": ('C=C=CC#C', 2021),
    "ethyl isocyanate": ('CCN=C=O', 2021),
    "cyclopentadiene": ('C1=CCC=C1', 2021),
    "cyanodiacetylene": ('C#CC#CC#N', 1976),
    "triacetylene": ('C#CC#CC#C', 2001),
    "benzene": ('c1ccccc1', 2001),
    "trans-cyanovinylacetylene": ('C#CC=CC#N', 2021),
    "vinylcyanoacetylene": ('C=CC#CC#N', 2021),
    "ortho-benzyne": ('C1#CC=CC=C1', 2021),
}
# In-range census molecules that can be written only with formally separated charges (dative bonds). They are absent
# from the neutral closed-shell space by construction (manuscript Section 3.1.2).
CHARGE_SEPARATED = {
    "carbon monoxide": "[C-]#[O+]",
    "hydrogen isocyanide HNC": "[C-]#[NH+]",
    "nitrous oxide": "[N-]=[N+]=O",
    "methyl isocyanide": "[C-]#[N+]C",
    "fulminic acid HCNO": "C#[N+][O-]",
    "isocyanoacetylene HCCNC": "C#C[N+]#[C-]",
    "isocyanogen CNCN": "N#C[N+]#[C-]",
    "isocyanodiacetylene HC4NC": "C#CC#C[N+]#[C-]",
}
# Molecules reported after the census cut-off (2022-2023). They are NOT positives: they stay in the undetected pool and
# are used only for the prospective check of the final ranking (M5, Table S3).
POST2021 = {
    "1,2-ethenediol": ('OC=CO', 2022),
    "n-propanol": ('CCCO', 2022),
    "isopropanol": ('CC(C)O', 2022),
    "crotononitrile": ('CC=CC#N', 2022),
    "methacrylonitrile": ('C=C(C)C#N', 2022),
    "allyl cyanide": ('C=CCC#N', 2022),
    "methyl ketene": ('CC=C=O', 2023),
    "E-1-cyano-1,3-butadiene": ('C=CC=CC#N', 2023),
}
# Molecular oxygen (triplet ground state) is represented by its closed-shell valence structure O=O and retained as a detection.
# Excluded as open-shell (CH2 and the radicals) or not representable with hydrogen-filled standard valences
# (C2, C3, C5, C2O, C3O, C5O, HC2N, HC4N, c-C3H2, l-C3H2, H2C4, H2C5, H2C6, HNCCC, C3HCCH, HCCCHCCC).

lab, nm, yr = {}, {}, {}
for k,(s,y) in ISM.items():
    m=Chem.MolFromSmiles(s)
    if m is None or m.GetNumHeavyAtoms()>MAX_HEAVY: continue
    c=canon(m); lab[c]=1; nm[c]=k; yr[c]=y
cs_in_range = {k:s for k,s in CHARGE_SEPARATED.items() if Chem.MolFromSmiles(s).GetNumHeavyAtoms()<=MAX_HEAVY}

present = set(space.smiles)
missing = [(c,k) for c,k in nm.items() if c not in present]
N_ISM_NEUTRAL, N_ISM_CS = len(nm), len(cs_in_range)          # kept for the export cell (nm is reused as a local name later)
print(f"Census molecules within <= {MAX_HEAVY} heavy atoms: {N_ISM_NEUTRAL+N_ISM_CS} "
      f"({len(nm)} neutral closed-shell valence structures + {len(cs_in_range)} charge-separated)")
print(f"  present in the enumerated space: {len(nm)-len(missing)} of {len(nm)}")
assert not missing, f"Confirmed molecules missing from the enumeration: {missing}"
print("\nDetected molecules absent from the neutral closed-shell space (manuscript Section 3.1.2):")
for k,s in cs_in_range.items():
    c=canon(Chem.MolFromSmiles(s)); inside = c in present
    print(f"   {k:32} {s:20} -> {'present (charge separation ON)' if inside else 'charge-separated: absent by construction'}")
    if inside and ALLOW_CHARGE_SEPARATION: lab[c]=1; nm[c]=k
if not ALLOW_CHARGE_SEPARATION:
    print(f"These {len(cs_in_range)} molecules can only be written with formally separated charges and are absent by construction.")

post_nm = {canon(Chem.MolFromSmiles(s)):k for k,(s,y) in POST2021.items() if Chem.MolFromSmiles(s).GetNumHeavyAtoms()<=MAX_HEAVY}
assert all(c in present for c in post_nm), "prospective molecules missing from the enumeration"
assert not (set(post_nm) & set(nm)), "a prospective molecule is also a positive"

space["is_ism"]   = space.smiles.map(lambda s: lab.get(s,0))
space["ism_name"] = space.smiles.map(lambda s: nm.get(s,""))
space["ism_year"] = space.smiles.map(lambda s: yr.get(s,0))
space["post2021"] = space.smiles.map(lambda s: post_nm.get(s,""))

print("\nper-stratum composition:")
print(space.assign(prospective=(space.post2021!="").astype(int)).groupby("heavy_atoms").agg(molecules=("smiles","size"), ism=("is_ism","sum"), prospective=("prospective","sum")).to_string())

# containment check against QM9, if the reference file is available
ref = None
if QM9_REF is not None:
    ref=pd.read_csv(QM9_REF)
    ref = ref[ref.heavy_atoms<=MAX_HEAVY].copy()
    # QM9 stores non-canonical SMILES (e.g. 'CC#C' vs 'C#CC'): canonicalise with the standard RDKit
    # round trip before merging. (canon() above is written for freshly built RWMol objects and
    # rejects external SMILES that carry aromatic [nH] atoms, which would silently drop the
    # NH-heterocycles of the reference.)
    def _std(s):
        m = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m) if m is not None else None
    ref["smiles"] = [_std(s) for s in ref.smiles]
    ref = ref.dropna(subset=["smiles"]).drop_duplicates("smiles")
    # completeness check, restricted to the element range of the enumeration
    ELEMS = {1,6,7,8} | ({9} if INCLUDE_FLUORINE else set())
    in_range = [all(a.GetAtomicNum() in ELEMS for a in Chem.MolFromSmiles(s).GetAtoms()) for s in ref.smiles]
    qs = set(ref[in_range].smiles); S_ = set(space.smiles)
    inside = qs & S_; outside = qs - S_
    QM9_RECOVERY_PCT = len(inside)/len(qs)*100
    print(f"\nQM9 reference molecules with <= {MAX_HEAVY} heavy atoms: {len(ref)}  ({len(qs)} within the element range)")
    print(f"  contained in AstroSpace: {len(inside)}  ({QM9_RECOVERY_PCT:.1f}%)")
    if outside:
        print(f"  not contained: {len(outside)} (zwitterionic structures, excluded by the neutral-charge rule):")
        for s in sorted(outside): print("    ", s)
    assert QM9_RECOVERY_PCT > 95, "Containment below 95%: check the canonicalisation of the reference SMILES."


space.to_csv(OUT/f"AstroSpace{MAX_HEAVY}_enumerated.csv", index=False)
print(f"\nsaved -> AstroSpace{MAX_HEAVY}_enumerated.csv   ({len(space):,} rows, {space.is_ism.sum()} positives)")

---
## M2 — GFN2-xTB properties

For each molecule, one initial geometry is generated with RDKit ETKDG (fixed random seed) and refined
with MMFF94, then optimized with GFN2-xTB [Bannwarth, Ehlert & Grimme, *J. Chem. Theory Comput.* **15**,
1652 (2019)] through `tblite`. The dipole moment, the rotational constants A ≥ B ≥ C (from the
moment-of-inertia tensor) and the total energy are taken from the optimized geometry; a single conformer
is retained per molecule. The computation is checkpointed every 500 molecules and can be resumed.
Molecules that fail geometry embedding or optimization, typically strained polycyclic cages, are
discarded and counted. The rotational partition function at 10 K and the line-intensity factor
μ²/Q_rot are computed from the rotational constants.


In [ ]:
# ============================================================
# CELL 5 — M2: GFN2-xTB dipole + rotational constants (checkpointed)
# ============================================================
from rdkit.Chem import AllChem
from ase import Atoms
from ase.optimize import BFGS
from tblite.ase import TBLite
from multiprocessing import Pool

AMU=1.66053906660e-27; ANG=1e-10; HPLANCK=6.62607015e-34

def rot_constants(masses, pos):
    """A >= B >= C in GHz from masses (amu) and coordinates (Angstrom)."""
    m=np.asarray(masses); r=np.asarray(pos)
    if len(m)<2: return (np.nan,)*3
    r = r - (m[:,None]*r).sum(0)/m.sum()
    I=np.zeros((3,3))
    for mi,ri in zip(m,r): I += mi*(np.dot(ri,ri)*np.eye(3) - np.outer(ri,ri))
    I *= AMU*ANG**2
    eig=np.sort(np.linalg.eigvalsh(I))                # Ia <= Ib <= Ic
    return tuple(HPLANCK/(8*np.pi**2*Ii)/1e9 if Ii>1e-49 else np.nan for Ii in eig)

def xtb_one(smi):
    try:
        m=Chem.AddHs(Chem.MolFromSmiles(smi))
        if AllChem.EmbedMolecule(m, randomSeed=SEED)!=0: return None
        AllChem.MMFFOptimizeMolecule(m, maxIters=400)
        conf=m.GetConformer()
        pos=np.array([list(conf.GetAtomPosition(i)) for i in range(m.GetNumAtoms())])
        at=Atoms(numbers=[a.GetAtomicNum() for a in m.GetAtoms()], positions=pos)
        with open(os.devnull,'w') as dn, contextlib.redirect_stdout(dn), contextlib.redirect_stderr(dn):
            at.calc = TBLite(method="GFN2-xTB", verbosity=0)
            BFGS(at, logfile=None).run(fmax=0.05, steps=XTB_STEPS)
            E   = at.get_potential_energy()
            dip = at.get_dipole_moment()
        A,B,C = rot_constants(at.get_masses(), at.get_positions())
        return dict(smiles=smi, mu_xtb=float(np.linalg.norm(dip)/0.2081943),
                    A=A, B=B, C=C, E_xtb=float(E), n_atoms=int(len(at)))
    except Exception:
        return None

todo = space.smiles.tolist()
done = pd.read_csv(CKPT) if CKPT.exists() else pd.DataFrame(columns=["smiles"])
done_set = set(done.smiles)
todo = [s for s in todo if s not in done_set]
print(f"to compute: {len(todo):,}  (already done: {len(done):,})")

if todo:
    t0=time.time(); results=list(done.to_dict("records")); CHUNK=500

    # Parallel map with a sequential fallback: multiprocessing needs picklable
    # top-level functions, which holds in a normal notebook run but can fail in
    # unusual execution contexts. If it does, we fall back to a serial loop so the
    # cell always completes rather than crashing.
    def _run_batch(batch):
        try:
            with Pool(N_PROC) as pool:
                out = pool.map(xtb_one, batch, chunksize=16)
        except Exception as e:
            print(f"  [parallel unavailable -> serial fallback: {type(e).__name__}]", flush=True)
            out = list(map(xtb_one, batch))
        # failures are recorded with NaN properties so that a resumed run does not retry them
        return [r if r else dict(smiles=s, mu_xtb=np.nan) for s, r in zip(batch, out)]

    for i in range(0, len(todo), CHUNK):
        batch = todo[i:i+CHUNK]
        results += _run_batch(batch)
        pd.DataFrame(results).to_csv(CKPT, index=False)
        n_ok = int(np.isfinite(pd.DataFrame(results).mu_xtb).sum())
        el=time.time()-t0; frac=(i+len(batch))/len(todo)
        print(f"  {i+len(batch):>7,}/{len(todo):,}  ok={n_ok:>7,}  "
              f"{el/60:.1f} min elapsed, ~{el/max(frac,1e-9)/60*(1-frac):.1f} min left", flush=True)

phys = pd.read_csv(CKPT).drop_duplicates("smiles")
phys = phys.merge(space, on="smiles", how="inner")
phys = phys[np.isfinite(phys.mu_xtb)]
# the checkpoint is written in completion order; sort so that the released file and every seeded
# random sample downstream are identical on every platform
phys = phys.sort_values(["heavy_atoms","smiles"], kind="mergesort").reset_index(drop=True)
print(f"\nxTB succeeded for {len(phys):,}/{len(space):,} molecules "
      f"({len(phys)/len(space)*100:.1f}%) | ISM positives retained: {int(phys.is_ism.sum())}")

K_OVER_H, T_CLOUD = 20.8367, 10.0     # GHz/K, TMC-1
def Qrot(a,b,c):
    if not np.all(np.isfinite([a,b,c])) or min(a,b,c)<=0: return np.nan
    return math.sqrt(math.pi*(K_OVER_H*T_CLOUD)**3/(a*b*c))
phys["Qrot"] = [Qrot(a,b,c) for a,b,c in zip(phys.A,phys.B,phys.C)]
phys["D_score"] = phys.mu_xtb**2 / phys.Qrot
phys.to_csv(OUT/f"AstroSpace{MAX_HEAVY}_xtb.csv", index=False)
print("saved ->", OUT/f"AstroSpace{MAX_HEAVY}_xtb.csv")

---
## M3 — Δ-ML dipole correction with conformal intervals (sensitivity analysis)

The GFN2-xTB dipole moments are compared with the B3LYP/6-31G(2df,p) values of QM9 for the molecules
common to both sets. A Δ-ML correction (gradient boosting on the semiempirical dipole moment and
fourteen cheap descriptors) toward the QM9 values is trained and assessed by five-fold cross-validation,
and prediction intervals are constructed by Mondrian split conformal prediction stratified by heavy-atom
count, with the empirical coverage measured over twenty repeated training, calibration and test splits.
The corrected values are evaluated as an additional predictor in M4; ranking and candidate selection
use the raw GFN2-xTB values throughout. The module requires `QM9_dipole_reference.csv`; if the file is
absent, the module is skipped and the notebook says so.


In [ ]:
# ============================================================
# CELL 6 — M3: Delta-ML + Mondrian conformal calibration
# ============================================================
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.ensemble import GradientBoostingRegressor
from scipy.stats import pearsonr, spearmanr

def dipole_features(smi, mu):
    m=Chem.MolFromSmiles(smi)
    return [mu, Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.NumHDonors(m),
            Descriptors.NumHAcceptors(m), Descriptors.NumRotatableBonds(m), Descriptors.RingCount(m),
            Descriptors.FractionCSP3(m), Descriptors.MolWt(m), rdMolDescriptors.CalcNumHeteroatoms(m),
            sum(a.GetAtomicNum()==7 for a in m.GetAtoms()), sum(a.GetAtomicNum()==8 for a in m.GetAtoms()),
            sum(a.GetAtomicNum()==9 for a in m.GetAtoms()),
            Descriptors.MaxPartialCharge(m) or 0, Descriptors.MinPartialCharge(m) or 0]

CALIBRATED = False
if QM9_REF is not None and ref is not None:
    ref_pairs = phys.merge(ref[["smiles","qm9_mu"]], on="smiles", how="inner").dropna(subset=["qm9_mu"])
    print(f"paired xTB / B3LYP molecules: {len(ref_pairs)}")
    if len(ref_pairs) >= 200:
        Xc = np.nan_to_num(np.array([dipole_features(s,mu) for s,mu in zip(ref_pairs.smiles, ref_pairs.mu_xtb)]))
        yc = ref_pairs.qm9_mu.values; sc_ = ref_pairs.heavy_atoms.values

        pr,_ = pearsonr(ref_pairs.mu_xtb, yc); sr,_ = spearmanr(ref_pairs.mu_xtb, yc)
        print(f"raw xTB : Pearson {pr:.3f}  Spearman {sr:.3f}  MAE {np.mean(abs(ref_pairs.mu_xtb-yc)):.3f} D")

        cv = cross_val_predict(GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,
                                                         subsample=0.8,random_state=SEED), Xc, yc, cv=5)
        pr2,_=pearsonr(cv,yc); sr2,_=spearmanr(cv,yc)
        print(f"Delta-ML: Pearson {pr2:.3f}  Spearman {sr2:.3f}  MAE {np.mean(abs(cv-yc)):.3f} D  (5-fold CV)")

        # Mondrian conformal coverage assessed over repeated splits
        covs=[]
        for s in range(20):
            tr,tmp = train_test_split(np.arange(len(yc)), test_size=.4, random_state=s)
            cal,te = train_test_split(tmp, test_size=.5, random_state=s)
            g = GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,
                                          subsample=0.8,random_state=s).fit(Xc[tr],yc[tr])
            res_c=np.abs(yc[cal]-g.predict(Xc[cal])); q={}
            for h in np.unique(sc_[cal]):
                rg=res_c[sc_[cal]==h]; n=len(rg)
                if n<15: continue
                q[h]=np.quantile(rg, min(np.ceil((n+1)*0.9)/n,1.0), method="higher")
            res_t=np.abs(yc[te]-g.predict(Xc[te])); hit=tot=0
            for h,qh in q.items():
                m_=sc_[te]==h
                if m_.sum()<3: continue
                hit+=np.sum(res_t[m_]<=qh); tot+=m_.sum()
            if tot: covs.append(hit/tot)
        print(f"Mondrian CP coverage over 20 splits: {np.mean(covs):.3f} +/- {np.std(covs):.3f}  (target 0.900)")

        # final model on all reference data -> apply to the whole space
        final = GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,
                                          subsample=0.8,random_state=SEED).fit(Xc,yc)
        res_all=np.abs(yc-final.predict(Xc))
        qhat={h: np.quantile(res_all[sc_==h], 0.9, method="higher")
              for h in np.unique(sc_) if (sc_==h).sum()>=15}
        Xa = np.nan_to_num(np.array([dipole_features(s,mu) for s,mu in zip(phys.smiles, phys.mu_xtb)]))
        phys["mu_cal"]   = final.predict(Xa)
        phys["mu_lo"]    = phys.mu_cal - phys.heavy_atoms.map(qhat)
        phys["mu_hi"]    = phys.mu_cal + phys.heavy_atoms.map(qhat)
        # D_score stays on the RAW dipole: calibration is for reporting, not ranking.
        phys["D_score"]  = phys.mu_xtb**2 / phys.Qrot
        RAW_MAE=float(np.mean(np.abs(ref_pairs.mu_xtb-ref_pairs.qm9_mu))); DML_MAE=float(np.mean(np.abs(cv-yc)))
        CALIBRATED = True
        print("\nCorrected dipole moments stored as mu_cal with 90% Mondrian conformal bounds [mu_lo, mu_hi]; ranking uses mu_xtb.")
        json.dump({"raw":{"pearson":float(pr),"spearman":float(sr)},
                   "delta_ml":{"pearson":float(pr2),"spearman":float(sr2),"mae":float(np.mean(abs(cv-yc)))},
                   "cp_coverage_mean":float(np.mean(covs)),"cp_coverage_sd":float(np.std(covs)),
                   "n_pairs":int(len(ref_pairs))}, open(OUT/"dipole_calibration.json","w"), indent=2)
if not CALIBRATED:
    if QM9_REF is None or ref is None:
        print("QM9_dipole_reference.csv not found: the Delta-ML sensitivity analysis (Section 2.6, Table 2 row 'Dipole moment, Delta-ML corrected', Figure S2c) is skipped. Add the file (see README) and re-run this cell to reproduce it.")
    else:
        print(f"Only {len(ref_pairs)} xTB/B3LYP pairs (< 200): too few to train the Delta-ML correction at this MAX_HEAVY; the sensitivity analysis is skipped.")
    phys["mu_cal"]=phys.mu_xtb; phys["mu_lo"]=np.nan; phys["mu_hi"]=np.nan

---
## M4 — Same-size evaluation

Each confirmed interstellar molecule is ranked only against undetected molecules of identical heavy-atom
count, which removes the size confound; the same-size AUC is the mean, over the detected molecules, of
the fraction of same-size undetected molecules ranked below each of them (ties count one half). A
size-only control scores exactly 0.500 by construction and serves as a diagnostic that the
stratification is correct. Tanimoto similarity and the descriptor model are evaluated leave-one-positive-out:
the held-out molecule never influences the model that ranks it. Significance against 0.500 is assessed
by a sign-flip permutation test over the detected molecules, with bootstrap confidence intervals, and
predictors are compared pairwise with the same permutation procedure on the per-molecule differences.


In [ ]:
# ============================================================
# CELL 7 — M4: size-matched retrieval, permutation tests, bootstrap CIs
# ============================================================
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import xgboost as xgb

y      = phys.is_ism.values
heavy  = phys.heavy_atoms.values
pos    = np.where(y==1)[0]
unl    = np.where(y==0)[0]
print(f"pool {len(phys):,} | positives {len(pos)} | prevalence {y.mean()*100:.2f}%")

def matched_auc(score_vec_or_map, min_neg=1):
    """Same-size AUC of each positive against the undetected molecules of its heavy-atom count (ties count one half; random = 0.5).
    Every positive with at least one same-size undetected molecule contributes (only the single-heavy-atom molecules are excluded)."""
    v=[]
    for p in pos:
        s = score_vec_or_map[p] if isinstance(score_vec_or_map, dict) else score_vec_or_map
        c = np.where((heavy==heavy[p]) & (y==0))[0]
        if len(c) < min_neg: continue
        v.append(np.mean(s[c] < s[p]) + 0.5*np.mean(s[c] == s[p]))
    return np.array(v)

def report(name, score):
    v = matched_auc(score)
    if len(v)==0: print(f"  {name:32s} (no stratum with enough negatives)"); return v
    g = np.random.default_rng(SEED)
    bs = [np.mean(g.choice(v,len(v),replace=True)) for _ in range(10000)]
    perm = [np.mean((v-0.5)*g.choice([-1,1],len(v))) for _ in range(10000)]
    p = np.mean(np.abs(perm) >= abs(v.mean()-0.5))
    print(f"  {name:32s} AUC={v.mean():.3f}  95%CI [{np.percentile(bs,2.5):.3f},{np.percentile(bs,97.5):.3f}]  p={p:.4f}  n={len(v)}")
    return v

# --- descriptors (size-free on purpose) ---
mols = [Chem.MolFromSmiles(s) for s in phys.smiles]
SF = np.array([[Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.NumHDonors(m),
                Descriptors.NumHAcceptors(m), Descriptors.NumRotatableBonds(m),
                Descriptors.RingCount(m), Descriptors.FractionCSP3(m)] for m in mols])
SF = np.nan_to_num(SF)

def loo_xgb(F, max_neg=20000):
    rng=np.random.default_rng(SEED); S={}
    for p in pos:
        tp = np.setdiff1d(pos,[p])
        neg = unl if len(unl)<=max_neg else rng.choice(unl, max_neg, replace=False)
        idx=np.concatenate([tp,neg]); yy=np.concatenate([np.ones(len(tp)),np.zeros(len(neg))])
        clf=xgb.XGBClassifier(n_estimators=150,max_depth=3,learning_rate=0.1,subsample=0.8,n_jobs=N_PROC,
              eval_metric='logloss',scale_pos_weight=len(neg)/max(len(tp),1),verbosity=0).fit(F[idx],yy)
        S[p]=clf.predict_proba(F)[:,1]
    return S

# --- Tanimoto to detected species, leave-one-out (no circular seed set) ---
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
def tanimoto_loo():
    S={}
    for p in pos:
        refs=[fps[i] for i in pos if i!=p]
        S[p]=np.array([max(DataStructs.BulkTanimotoSimilarity(f,refs)) for f in fps])
    return S

print("\n=== SAME-SIZE AUC (random = 0.500) ===")
v_ctrl = report("size-only (control)",        -heavy.astype(float))
assert abs(v_ctrl.mean()-0.5) < 1e-9, "stratification is broken: control must be exactly 0.500"
v_mu   = report("dipole mu (raw xTB)",         phys.mu_xtb.values)
v_D    = report("detectability mu^2/Qrot",     np.nan_to_num(phys.D_score.values, nan=-1))
if CALIBRATED:
    v_muc = report("dipole mu (Delta-ML cal.)", phys.mu_cal.values)
v_E    = report("xTB energy per atom",        -(phys.E_xtb/phys.n_atoms).values)
v_tan  = report("Tanimoto to detections (LOO)", tanimoto_loo())
v_xgb  = report("XGBoost, size-free desc (LOO)", loo_xgb(SF))

def paired(a,b,na,nb):
    n=min(len(a),len(b)); d_=a[:n]-b[:n]; g=np.random.default_rng(5)
    perm=[np.mean(d_*g.choice([-1,1],len(d_))) for _ in range(10000)]
    print(f"  {na:26s} - {nb:26s} delta={d_.mean():+.3f}  p={np.mean(np.abs(perm)>=abs(d_.mean())):.4f}")
print("\n=== PAIRED COMPARISONS ===")
paired(v_mu, v_ctrl, "dipole", "size-control")
paired(v_D,  v_mu,   "mu^2/Qrot", "dipole alone")
paired(v_xgb,v_mu,   "XGBoost", "dipole")
paired(v_xgb,v_tan,  "XGBoost", "Tanimoto")

json.dump({k: dict(auc=float(v.mean()), n=int(len(v))) for k,v in
           dict(size_control=v_ctrl, dipole=v_mu, detectability=v_D, xtb_energy=v_E,
                tanimoto=v_tan, xgboost=v_xgb).items()},
          open(OUT/"size_matched_results.json","w"), indent=2)

---
## M4b — Counterfactual size experiment and graph isomorphism network

The counterfactual experiment reports the whole-dataset and same-size AUC of the same leave-one-positive-out models, and repeats the descriptor model with heavy-atom count added as an eighth input (Table 1 of the main text). A bond-type-aware graph isomorphism network is then trained on graph connectivity alone under the identical protocol (Table 2, Figures 3–4, S4c and S5).

In [ ]:
# ============================================================
# CELL 7b — M4b: (i) counterfactual size experiment, (ii) graph isomorphism network
# under the identical leave-one-positive-out, same-size protocol (raw graph only).
# Requires jax (pre-installed on Colab/Kaggle). ~25 min on 2 CPU cores at MAX_HEAVY=6.
# ============================================================
import jax, jax.numpy as jnp, time
from rdkit.Chem import rdFingerprintGenerator
d2 = phys[phys.heavy_atoms>=2].reset_index(drop=True)      # AstroSpace analysed set (2-6 heavy atoms)
y2=d2.is_ism.values; h2=d2.heavy_atoms.values; pos2=np.where(y2==1)[0]; unl2=np.where(y2==0)[0]
mols2=[Chem.MolFromSmiles(s) for s in d2.smiles]
SF2=np.nan_to_num(np.array([[Descriptors.MolLogP(m),Descriptors.TPSA(m),Descriptors.NumHDonors(m),Descriptors.NumHAcceptors(m),
                              Descriptors.NumRotatableBonds(m),Descriptors.RingCount(m),Descriptors.FractionCSP3(m)] for m in mols2]))
def same_size(Sd):
    return np.array([np.mean(Sd[p][(h2==h2[p])&(y2==0)]<Sd[p][p])+0.5*np.mean(Sd[p][(h2==h2[p])&(y2==0)]==Sd[p][p]) for p in pos2])
def whole(Sd):
    return np.array([np.mean(Sd[p][unl2]<Sd[p][p])+0.5*np.mean(Sd[p][unl2]==Sd[p][p]) for p in pos2])
def as_dict(vec): return {p:vec for p in pos2}
rng=np.random.default_rng(42)
def loo_xgb2(F):
    Sd={}
    for p in pos2:
        tp=np.setdiff1d(pos2,[p]); neg=rng.choice(unl2,min(20000,len(unl2)),replace=False)
        idx=np.concatenate([tp,neg]); yy=np.concatenate([np.ones(len(tp)),np.zeros(len(neg))])
        clf=xgb.XGBClassifier(n_estimators=150,max_depth=3,learning_rate=0.1,subsample=0.8,n_jobs=N_PROC,eval_metric='logloss',
                              scale_pos_weight=len(neg)/len(tp),verbosity=0).fit(F[idx],yy); Sd[p]=clf.predict_proba(F)[:,1]
    return Sd
# ---- (i) counterfactual: whole-dataset vs same-size AUC of the SAME models; XGBoost with size added ----
print("=== COUNTERFACTUAL (Table 1): whole-dataset vs same-size AUC ===")
CF={}
CF['size_only']=(whole(as_dict(-h2.astype(float))).mean(), same_size(as_dict(-h2.astype(float))).mean())
CF['dipole']=(whole(as_dict(d2.mu_xtb.values)).mean(), same_size(as_dict(d2.mu_xtb.values)).mean())
gen2=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048); fps2=[gen2.GetFingerprint(m) for m in mols2]
Tn={}
for p in pos2:
    refs=[fps2[i] for i in pos2 if i!=p]; Tn[p]=np.array([max(DataStructs.BulkTanimotoSimilarity(f,refs)) for f in fps2])
CF['tanimoto']=(whole(Tn).mean(), same_size(Tn).mean())
X7=loo_xgb2(SF2); CF['xgb_7desc']=(whole(X7).mean(), same_size(X7).mean())
X8=loo_xgb2(np.column_stack([SF2,h2])); CF['xgb_7desc_plus_size']=(whole(X8).mean(), same_size(X8).mean())
for k,(w,s) in CF.items(): print(f"  {k:<22} whole-dataset={w:.3f}   same-size={s:.3f}")
from scipy.stats import spearmanr
bits=np.array([f.GetNumOnBits() for f in fps2]); print(f"  Morgan set-bit count vs heavy atoms: Spearman rho = {spearmanr(bits,h2)[0]:.3f}")
for nm,col in zip(['MolLogP','TPSA','NumHDonors','NumHAcceptors','NumRotatableBonds','RingCount','FractionCSP3'],SF2.T):
    print(f"  {nm:<18} rho = {spearmanr(col,h2)[0]:+.3f}")
# ---- (ii) GIN (GINE-style, bond-type-aware), manual Adam, no external optimiser ----
NMAX=6; ATOMS={'C':0,'N':1,'O':2}; BT={Chem.BondType.SINGLE:0,Chem.BondType.DOUBLE:1,Chem.BondType.TRIPLE:2,Chem.BondType.AROMATIC:3}; F=13
def featurize(m):
    X=np.zeros((NMAX,F),np.float32); A=np.zeros((4,NMAX,NMAX),np.float32); M=np.zeros(NMAX,np.float32)
    for a in m.GetAtoms():
        i=a.GetIdx(); X[i,ATOMS[a.GetSymbol()]]=1; X[i,3+min(a.GetDegree(),4)-1]=1; X[i,7+min(a.GetTotalNumHs(),3)]=1
        X[i,11]=float(a.GetIsAromatic()); X[i,12]=float(a.IsInRing()); M[i]=1
    for b in m.GetBonds():
        i,j=b.GetBeginAtomIdx(),b.GetEndAtomIdx(); t=BT[b.GetBondType()]; A[t,i,j]=1; A[t,j,i]=1
    return X,A,M
feats=[featurize(m) for m in mols2]; GX=np.stack([f[0] for f in feats]); GA=np.stack([f[1] for f in feats]); GM=np.stack([f[2] for f in feats])
HID,L,EPOCHS,BS,LR=64,3,30,512,1e-3
N_GIN_SEEDS=3      # networks per fold (independent initialisation and minibatch order); their scores are averaged
def init(key):
    ks=jax.random.split(key,12); p={}
    def lin(k,i,o): return {'W':jax.random.normal(k,(i,o))*np.sqrt(2/i),'b':jnp.zeros(o)}
    p['emb']=lin(ks[0],F,HID); p['bond']=jax.random.normal(ks[1],(4,HID))*0.1; p['eps']=jnp.zeros(L)
    for l in range(L): p[f'mlp{l}a']=lin(ks[2+2*l],HID,HID); p[f'mlp{l}b']=lin(ks[3+2*l],HID,HID)
    p['out1']=lin(ks[10],HID*(L+1),HID); p['out2']=lin(ks[11],HID,1); return p
def fwd(p,X,A,M):
    H=jax.nn.relu(X@p['emb']['W']+p['emb']['b'])*M[...,None]; reads=[H.sum(1)]
    for l in range(L):
        msg=sum(A[:,t]@jax.nn.relu(H+p['bond'][t]) for t in range(4)); H=(1+p['eps'][l])*H+msg
        H=jax.nn.relu(H@p[f'mlp{l}a']['W']+p[f'mlp{l}a']['b']); H=jax.nn.relu(H@p[f'mlp{l}b']['W']+p[f'mlp{l}b']['b'])*M[...,None]; reads.append(H.sum(1))
    z=jax.nn.relu(jnp.concatenate(reads,-1)@p['out1']['W']+p['out1']['b']); return (z@p['out2']['W']+p['out2']['b'])[:,0]
def loss(p,X,A,M,yy,pw):
    z=fwd(p,X,A,M); return jnp.mean(pw*yy*jax.nn.softplus(-z)+(1-yy)*jax.nn.softplus(z))
def adam_init(p): return {'m':jax.tree_util.tree_map(jnp.zeros_like,p),'v':jax.tree_util.tree_map(jnp.zeros_like,p),'t':0}
@jax.jit
def step(p,st,X,A,M,yy,pw):
    l,g=jax.value_and_grad(loss)(p,X,A,M,yy,pw); t=st['t']+1
    m=jax.tree_util.tree_map(lambda m_,g_:0.9*m_+0.1*g_,st['m'],g); v=jax.tree_util.tree_map(lambda v_,g_:0.999*v_+0.001*g_*g_,st['v'],g)
    upd=jax.tree_util.tree_map(lambda m_,v_:LR*(m_/(1-0.9**t))/(jnp.sqrt(v_/(1-0.999**t))+1e-8),m,v)
    return jax.tree_util.tree_map(lambda p_,u_:p_-u_,p,upd),{'m':m,'v':v,'t':t},l
predict=jax.jit(lambda p,X,A,M: jax.nn.sigmoid(fwd(p,X,A,M)))
def train_gin(seed,idx):
    p=init(jax.random.PRNGKey(seed)); st=adam_init(p); yy=y2[idx].astype(np.float32); pw=float((yy==0).sum()/(yy==1).sum()); r=np.random.default_rng(seed); curve=[]
    for ep in range(EPOCHS):
        perm=r.permutation(len(idx)); ls=[]
        for b in range(0,len(idx),BS):
            bi=idx[perm[b:b+BS]]; p,st,l=step(p,st,jnp.asarray(GX[bi]),jnp.asarray(GA[bi]),jnp.asarray(GM[bi]),jnp.asarray(y2[bi].astype(np.float32)),pw); ls.append(float(l))
        curve.append(float(np.mean(ls)))
    return p,curve
def score_all(p): return np.concatenate([np.asarray(predict(p,jnp.asarray(GX[b:b+4096]),jnp.asarray(GA[b:b+4096]),jnp.asarray(GM[b:b+4096]))) for b in range(0,len(d2),4096)])
print("\n=== GIN under the identical leave-one-positive-out, same-size protocol ===")
S_gin={}; gin_curves=[]; t0=time.time()
for k,p_ in enumerate(pos2):
    r=np.random.default_rng(42+k); tp=np.setdiff1d(pos2,[p_]); neg=r.choice(unl2,min(20000,len(unl2)),replace=False)
    idx=np.concatenate([tp,neg]); scores=[]; finals=[]
    for s_ in range(N_GIN_SEEDS):
        prm,curve=train_gin(1000+k+100*s_,idx); scores.append(score_all(prm)); gin_curves.append(curve); finals.append(curve[-1])
    S_gin[p_]=np.mean(scores,axis=0)
    if k%10==0: print(f"  fold {k+1}/{len(pos2)}  final loss of the {N_GIN_SEEDS} networks: {', '.join(f'{f:.3f}' for f in finals)}  {time.time()-t0:.0f}s",flush=True)
v_gin=same_size(S_gin); g=np.random.default_rng(3)
ci=np.percentile([np.mean(g.choice(v_gin,len(v_gin),replace=True)) for _ in range(10000)],[2.5,97.5]); perm=[np.mean((v_gin-0.5)*g.choice([-1,1],len(v_gin))) for _ in range(10000)]
print(f"GIN same-size AUC = {v_gin.mean():.3f}  95%CI [{ci[0]:.3f},{ci[1]:.3f}]  p={np.mean(np.abs(perm)>=abs(v_gin.mean()-0.5)):.4f}   whole-dataset AUC = {whole(S_gin).mean():.3f}")
v_xgb2=same_size(X7); v_tan2=same_size(Tn)
for nm,other in [('Tanimoto',v_tan2),('XGBoost',v_xgb2)]:
    dd=v_gin-other; pp=[np.mean(dd*g.choice([-1,1],len(dd))) for _ in range(10000)]; print(f"  GIN - {nm}: delta={dd.mean():+.3f}  p={np.mean(np.abs(pp)>=abs(dd.mean())):.4f}")
json.dump({'counterfactual':{k:[float(a),float(b)] for k,(a,b) in CF.items()},'gin_same':float(v_gin.mean()),'gin_ci':[float(ci[0]),float(ci[1])]},open(OUT/"gin_and_counterfactual.json","w"),indent=2)


---
## M5 — Candidate prioritization and prospective check

Undetected molecules are scored with the descriptor model (mean of the leave-one-positive-out models,
rescaled to [0, 1]). Molecules in the highest 5% of GFN2-xTB energy per atom are removed as a crude
strain screen, and a GFN2-xTB dipole moment above 0.1 D is required. The 50 highest-scoring molecules
(Table S1) and the five highest-scoring molecules within each heavy-atom count (Table S2) are written
with their dipole moments, rotational constants and scores. As a prospective check, the molecules
reported after the census cut-off (2022–2023), which are not positives anywhere in this notebook, are
located in the ranking of the undetected pool (Table S3).


In [ ]:
# ============================================================
# CELL 8 — M5: ranked, filtered candidates and prospective check
# ============================================================
S_final = loo_xgb(SF)
ml = np.mean([S_final[p] for p in pos], axis=0)          # ensemble of the LOO models
ml = (ml - ml.min())/(np.ptp(ml)+1e-12)

cand = phys.copy()
cand["ml_score"]   = ml
cand["E_per_atom"] = cand.E_xtb / cand.n_atoms
stab_cut = cand.E_per_atom.quantile(0.95)                 # drop the least stable 5%
cand["plausible"]  = (cand.E_per_atom < stab_cut) & (cand.mu_xtb > 0.1)   # raw dipole gates visibility

ranked = (cand[(cand.plausible) & (cand.is_ism==0)]
          .sort_values("ml_score", ascending=False).head(50)
          [["smiles","heavy_atoms","mu_xtb","mu_cal","mu_lo","mu_hi","A","B","C","Qrot","ml_score","E_per_atom"]]
          .round(4).reset_index(drop=True))
ranked.index += 1
ranked.to_csv(OUT/"top50_novel_candidates.csv")

recovered = cand[(cand.is_ism==1)].sort_values("ml_score", ascending=False)
print("Highest-scoring confirmed interstellar molecules (positive controls):")
print(recovered.head(10)[["smiles","ism_name","mu_xtb","ml_score"]].round(3).to_string(index=False))
print("\nTop-50 undetected candidates written to top50_novel_candidates.csv")
print(f"Candidates whose conformal lower bound mu_lo < 0.1 D (uncertain detectability): "
      f"{int((ranked.mu_lo < 0.1).sum())}/50")
ranked.head(15)

# ---- Supplementary tables: Table S1 (top-50, pooled) and Table S2 (top-5 within each heavy-atom count) ----
from rdkit.Chem import rdMolDescriptors as _rdmd
_fmt=lambda smi: _rdmd.CalcMolFormula(Chem.MolFromSmiles(smi))
tS1=ranked.head(50).copy(); tS1.insert(0,"rank",range(1,len(tS1)+1)); tS1["formula"]=[_fmt(s) for s in tS1.smiles]
tS1[["rank","smiles","formula","heavy_atoms","mu_xtb","B","ml_score"]].round(3).to_csv(OUT/"TableS1_top50_candidates.csv",index=False)
_pool=cand[(cand.plausible)&(cand.is_ism==0)]
tS2=pd.concat([_pool[_pool.heavy_atoms==n].sort_values("ml_score",ascending=False).head(5).assign(rank_in_stratum=range(1,min(5,(_pool.heavy_atoms==n).sum())+1)) for n in sorted(_pool.heavy_atoms.unique())])
tS2["formula"]=[_fmt(s) for s in tS2.smiles]
tS2[["heavy_atoms","rank_in_stratum","smiles","formula","mu_xtb","B","ml_score"]].round(3).to_csv(OUT/"TableS2_top5_per_stratum.csv",index=False)
print("Table S1 / Table S2 written:", OUT/"TableS1_top50_candidates.csv", OUT/"TableS2_top5_per_stratum.csv")

# ---- Prospective check (Table S3): molecules reported after the census cut-off, ranked as undetected molecules ----
_und = cand[(cand.is_ism==0)&(cand.heavy_atoms>=2)].copy()                 # the undetected pool used throughout
_und["rank_score"] = _und.ml_score.rank(ascending=False, method="min").astype(int)
_und["rank_dipole"] = _und.mu_xtb.rank(ascending=False, method="min").astype(int)
_rows=[]
for smi,name in post_nm.items():
    if smi not in set(_und.smiles): continue
    r=_und[_und.smiles==smi].iloc[0]; same=_und[_und.heavy_atoms==r.heavy_atoms]
    _rows.append(dict(name=name, smiles=smi, formula=_fmt(smi), heavy_atoms=int(r.heavy_atoms), year=POST2021[name][1],
        mu_xtb=round(float(r.mu_xtb),3), score=round(float(r.ml_score),3),
        rank_score=int(r.rank_score), pool=int(len(_und)), pct_score=round(float((_und.ml_score<r.ml_score).mean()*100),1),
        rank_in_stratum=int((same.ml_score>r.ml_score).sum()+1), stratum_size=int(len(same)),
        pct_same_size=round(float((same.ml_score<r.ml_score).mean()*100),1),
        rank_dipole=int(r.rank_dipole), pct_dipole=round(float((_und.mu_xtb<r.mu_xtb).mean()*100),1),
        in_top50=bool(smi in set(ranked.smiles))))
tS3=pd.DataFrame(_rows).sort_values("rank_score"); tS3.to_csv(OUT/"TableS3_prospective.csv",index=False)
print("\nProspective check: molecules reported in 2022-2023, ranked among the", len(_und), "undetected molecules")
print(tS3[["name","smiles","heavy_atoms","year","score","rank_score","pct_score","rank_in_stratum","stratum_size","pct_dipole","in_top50"]].to_string(index=False))


---
## M6 — Figures 1–5

Heavy-atom count distributions and whole-dataset ranking on QM9 (Figure 1), dipole-moment distributions
of the detected and undetected populations (Figure 2), same-size AUC of the predictors (Figure 3),
receiver operating characteristic curves from same-size pairs (Figure 4) and the twelve highest-scoring
candidates as structures (Figure 5).


In [ ]:
# ============================================================
# CELL 9 - M6: main-text figures 1-5 (300 dpi PNG + PDF). Descriptive panel titles only.
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from rdkit.Chem import Draw, rdMolDescriptors
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"savefig.bbox":"tight","font.family":"DejaVu Sans","font.size":10.5,
  "axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":0.25,"axes.axisbelow":True})
GREY='#8C8C8C'; SIZECOL={2:WONG['purple'],3:WONG['blue'],4:WONG['green'],5:WONG['orange'],6:WONG['yellow']}
def save(name): plt.savefig(FIG/f'{name}.png'); plt.savefig(FIG/f'{name}.pdf'); plt.show()
CAL = 'mu_cal' in d2.columns and CALIBRATED
v_cal = same_size(as_dict(d2.mu_cal.values)) if CAL else None
def ci_of(v): return np.percentile([np.mean(g.choice(v,len(v),replace=True)) for _ in range(10000)],[2.5,97.5])
def p_of(v): return np.mean(np.abs([np.mean((v-0.5)*g.choice([-1,1],len(v))) for _ in range(10000)])>=abs(v.mean()-0.5))
# ---- Figure 1: size distributions (ISM counts annotated) + QM9 whole-dataset AUC ----
fig,ax=plt.subplots(1,2,figsize=(11.5,4.4))
qm9={1:3,2:5,3:8,4:23,5:130,6:1193,7:7639,8:21444,9:103383}   # QM9 molecules per heavy-atom count (Ramakrishnan et al. 2014)
astro=phys.groupby('heavy_atoms').size().to_dict(); ism=phys[phys.is_ism==1].groupby('heavy_atoms').size().to_dict()
ax[0].bar([x-0.22 for x in sorted(qm9)],[qm9[x] for x in sorted(qm9)],width=.42,color=WONG['vermillion'],edgecolor='k',lw=.4,alpha=.9,label='QM9')
ax[0].bar([x+0.22 for x in sorted(astro)],[astro[x] for x in sorted(astro)],width=.42,color=WONG['skyblue'],edgecolor='k',lw=.4,alpha=.9,label='AstroSpace (this work)')
for x in sorted(ism):
    ax[0].scatter(x,ism[x],marker='*',s=170,color=WONG['orange'],edgecolor='k',lw=.6,zorder=5)
    ax[0].annotate(f'{ism[x]}',(x,ism[x]),textcoords='offset points',xytext=(0,9),ha='center',fontsize=9,fontweight='bold',color='#7a4a00')
ax[0].scatter([],[],marker='*',s=130,color=WONG['orange'],edgecolor='k',lw=.6,label='Confirmed interstellar species (count shown)')
ax[0].set_yscale('log'); ax[0].set_xlabel('Heavy atoms'); ax[0].set_ylabel('Number of molecules (log scale)'); ax[0].set_title('(a) Heavy-atom count distributions',fontsize=11)
ax[0].legend(fontsize=8.5,loc='upper left',framealpha=.95); ax[0].set_xticks(range(1,10)); ax[0].set_ylim(0.7,3e5)
# (b) cumulative size distributions, and the whole-dataset AUC that heavy-atom count alone obtains on each pool.
# On QM9 the value follows from its heavy-atom distribution and the sizes of the confirmed molecules (a size-only
# ranker orders molecules by heavy-atom count; ties count one half); on AstroSpace it is computed directly (Table 1).
sizes=np.arange(1,10); n_q=np.array([qm9.get(n,0) for n in sizes],float); n_a=np.array([astro.get(n,0) for n in sizes],float)
ism_sizes=phys[phys.is_ism==1].heavy_atoms.values
def size_only_auc(pool_counts):
    tot=pool_counts.sum(); return float(np.mean([(pool_counts[sizes>s].sum()+0.5*pool_counts[sizes==s].sum())/tot for s in ism_sizes]))
QM9_SIZE_ONLY_AUC=size_only_auc(n_q); ASTRO_SIZE_ONLY_AUC=size_only_auc(n_a)
cum=lambda v: np.cumsum(v)/v.sum()*100
ax[1].plot(sizes,cum(n_q),'o-',color=WONG['vermillion'],lw=2,ms=6,label='QM9')
ax[1].plot(sizes,cum(n_a),'s-',color=WONG['skyblue'],lw=2,ms=6,label='AstroSpace (this work)')
n_i=np.array([(ism_sizes==n).sum() for n in sizes],float); ax[1].plot(sizes,cum(n_i),'*-',color=WONG['orange'],lw=2,ms=11,mec='k',mew=.5,label='Confirmed interstellar molecules (in range)')
ax[1].set_xticks(sizes); ax[1].set_xlabel('Heavy atoms'); ax[1].set_ylabel('Cumulative share of molecules (%)'); ax[1].set_ylim(0,104)
ax[1].set_title('(b) Cumulative heavy-atom count distributions',fontsize=11); ax[1].legend(fontsize=8.5,loc='center right',framealpha=.95)
ax[1].text(0.03,0.93,f'Whole-dataset AUC of heavy-atom count alone:\nQM9 {QM9_SIZE_ONLY_AUC:.3f}   AstroSpace {ASTRO_SIZE_ONLY_AUC:.3f}',transform=ax[1].transAxes,fontsize=8.5,va='top',bbox=dict(boxstyle='round',fc='white',ec='grey',alpha=.9))
print(f"size-only whole-dataset AUC: QM9 {QM9_SIZE_ONLY_AUC:.3f} | AstroSpace {ASTRO_SIZE_ONLY_AUC:.3f}")
plt.tight_layout(); save('Figure1_size_distributions')
# ---- Figure 2: dipole moments pooled + within stratum ----
fig,ax=plt.subplots(1,2,figsize=(11.5,4.4))
ism_mu=d2[d2.is_ism==1].mu_xtb.values; non_mu=d2[d2.is_ism==0].mu_xtb.values; bins=np.linspace(0,6.5,27)
ax[0].hist(non_mu,bins=bins,density=True,alpha=.55,color=WONG['skyblue'],edgecolor='k',lw=.3,label=f'Undetected (n = {len(non_mu):,})')
ax[0].hist(ism_mu,bins=bins,density=True,alpha=.7,color=WONG['orange'],edgecolor='k',lw=.3,label=f'Confirmed interstellar (n = {len(ism_mu)})')
ax[0].axvline(np.median(non_mu),color=WONG['blue'],ls=':',lw=1.6,label=f'median {np.median(non_mu):.2f} D'); ax[0].axvline(np.median(ism_mu),color=WONG['vermillion'],ls='--',lw=1.6,label=f'median {np.median(ism_mu):.2f} D')
ax[0].set_xlabel('GFN2-xTB dipole moment μ (D)'); ax[0].set_ylabel('Probability density'); ax[0].set_title('(a) All heavy-atom counts pooled',fontsize=11); ax[0].legend(fontsize=8.5)
strata=[n for n in [2,3,4,5,6] if (d2[(d2.heavy_atoms==n)&(d2.is_ism==1)].shape[0]>5)]; pp=[]; lab=[]
for k,n in enumerate(strata):
    im=d2[(d2.heavy_atoms==n)&(d2.is_ism==1)].mu_xtb.values; nm=d2[(d2.heavy_atoms==n)&(d2.is_ism==0)].mu_xtb.values
    b1=ax[1].boxplot([nm],positions=[k*3+0.6],widths=.5,patch_artist=True,showfliers=False,medianprops=dict(color='k',lw=1.3)); b2=ax[1].boxplot([im],positions=[k*3+1.4],widths=.5,patch_artist=True,showfliers=False,medianprops=dict(color='k',lw=1.3))
    b1['boxes'][0].set_facecolor(WONG['skyblue']); b1['boxes'][0].set_alpha(.7); b2['boxes'][0].set_facecolor(WONG['orange']); b2['boxes'][0].set_alpha(.8)
    ax[1].scatter(np.full(len(im),k*3+1.4)+g.uniform(-.12,.12,len(im)),im,s=18,color=WONG['vermillion'],edgecolor='k',lw=.3,zorder=5,alpha=.85); pp.append(k*3+1); lab.append(f'{n} heavy atoms\n{len(im)} detected\n{len(nm):,} undetected')
ax[1].set_xticks(pp); ax[1].set_xticklabels(lab,fontsize=8); ax[1].set_ylabel('GFN2-xTB dipole moment μ (D)'); ax[1].set_title('(b) Within each heavy-atom count',fontsize=11)
ax[1].legend(handles=[Patch(facecolor=WONG['skyblue'],alpha=.7,label='Undetected'),Patch(facecolor=WONG['orange'],alpha=.8,label='Confirmed interstellar')],fontsize=8.5,loc='upper left')
plt.tight_layout(); save('Figure2_dipole_characterisation')
# ---- Figure 3: same-size AUC bars ----
# The Delta-ML-corrected dipole (sensitivity analysis) is reported in Table 2 and Figure S2c only.
names=['Size-only\ncontrol','Dipole μ\n(GFN2-xTB)','μ²/Q$_{rot}$','Tanimoto\nsimilarity','GIN\n(graph only)','XGBoost\n(7 descriptors)']
vecs=[v_ctrl,v_mu,v_D,v_tan,v_gin,v_xgb]
cols=[GREY,WONG['green'],WONG['skyblue'],WONG['blue'],WONG['purple'],WONG['orange']]
auc=[v.mean() for v in vecs]; cis=[ci_of(v) for v in vecs]; pv=[p_of(v) for v in vecs]
fig,ax=plt.subplots(figsize=(10.2,4.9)); err=[np.array(auc)-np.array([c[0] for c in cis]),np.array([c[1] for c in cis])-np.array(auc)]
bars=ax.bar(names,auc,yerr=err,capsize=5,color=cols,edgecolor='#333',lw=.6,alpha=.92,width=.64,error_kw=dict(lw=1.1,capthick=1.1,ecolor='#333'))
ax.axhline(0.5,ls='--',color='grey',lw=1.2)
for i,(b,v) in enumerate(zip(bars,auc)):
    ax.text(b.get_x()+b.get_width()/2,cis[i][1]+0.018,f'{v:.3f}',ha='center',fontsize=9.5,fontweight='bold',color='black')
    ax.text(b.get_x()+b.get_width()/2,0.44,'p < 0.001' if pv[i]<0.001 else f'p = {pv[i]:.3f}',ha='center',fontsize=8,color='black')
ax.set_ylim(0.42,0.99); ax.set_ylabel('Same-size AUC (0.5 = random)'); plt.tight_layout(); save('Figure3_same_size_auc')
# ---- Figure 4: ROC from same-size pairs ----
def roc(getter):
    pairs=[]
    for k,p in enumerate(pos2):
        c=np.where((h2==h2[p])&(y2==0))[0]; s=getter(k,p); pairs+= [(s[p],s[ci]) for ci in c]
    pairs=np.array(pairs); ths=np.sort(np.unique(np.concatenate([pairs[:,0],pairs[:,1]])))[::-1]
    f=np.array([0]+[np.mean(pairs[:,1]>=t) for t in ths]+[1]); t=np.array([0]+[np.mean(pairs[:,0]>=t) for t in ths]+[1]); o=np.argsort(f); return f[o],t[o]
curves=[('XGBoost (7 descriptors)',lambda k,p:X7[p],WONG['orange'],v_xgb.mean()),('Tanimoto similarity',lambda k,p:Tn[p],WONG['blue'],v_tan.mean()),
        ('GIN (graph only)',lambda k,p:S_gin[p],WONG['purple'],v_gin.mean()),('Dipole μ (GFN2-xTB)',lambda k,p:d2.mu_xtb.values,WONG['green'],v_mu.mean())]
curves.append(('μ²/Q$_{rot}$',lambda k,p:np.nan_to_num(d2.D_score.values,nan=-1),WONG['skyblue'],v_D.mean()))
fig,ax=plt.subplots(figsize=(6.4,5.8))
for nm,fn,col,a in curves: f,t=roc(fn); ax.plot(f,t,color=col,lw=2.1,label=f'{nm} (AUC {a:.2f})')
ax.plot([0,1],[0,1],'--',color='grey',lw=1.2,label='Random (AUC 0.50)'); ax.set_xlabel('False-positive rate (same-size pairs)'); ax.set_ylabel('True-positive rate (same-size pairs)')
ax.legend(fontsize=8.6,loc='lower right',framealpha=.95); ax.set_xlim(-.02,1.02); ax.set_ylim(-.02,1.02); plt.tight_layout(); save('Figure4_roc')
# ---- Figure 5: candidate structures with key quantities ----
top12=ranked.head(12).reset_index(drop=True); ncol,nrow=4,3
fig,axs=plt.subplots(nrow,ncol,figsize=(ncol*2.7,nrow*2.75))
for k in range(nrow*ncol):
    a=axs[k//ncol][k%ncol]; a.axis('off')
    if k>=len(top12): continue
    r=top12.iloc[k]; m=Chem.MolFromSmiles(r.smiles); a.imshow(Draw.MolToImage(m,size=(240,170)))
    a.set_title(f"{k+1}. {rdMolDescriptors.CalcMolFormula(m)}  ({int(r.heavy_atoms)} heavy atoms)",fontsize=8.6,pad=2)
    a.text(0.5,-0.06,f"μ(xTB) = {r.mu_xtb:.2f} D\nscore = {r.ml_score:.3f}   B = {r.B:.1f} GHz",transform=a.transAxes,ha='center',va='top',fontsize=7.8,color='#333')
plt.subplots_adjust(hspace=.55,wspace=.15); plt.savefig(FIG/'Figure5_candidates.png',dpi=300,bbox_inches='tight',facecolor='white'); plt.savefig(FIG/'Figure5_candidates.pdf',bbox_inches='tight',facecolor='white'); plt.show()
print("Figures 1-5 written to",FIG)


---
## M7 — Supplementary figures and graphical abstract

Enumeration completeness (S1), control experiments on charge separation, fluorine and the Δ-ML
correction (S2), the joint distribution of dipole moment and rotational constant B (S3), the size
dependence of descriptors and fingerprints together with the GIN training curves (S4), pairwise
permutation p values (S5), and the graphical abstract. The charge-separation and fluorine values shown
in Figure S2a,b come from the corresponding control runs of this notebook (configuration flags).


In [ ]:
# ============================================================
# CELL 9b - M7: supplementary figures S1-S5 and graphical abstract
# ============================================================
# S1 enumeration completeness
fig,ax=plt.subplots(1,2,figsize=(11.5,4.3)); counts=space.groupby('heavy_atoms').size()
ax[0].bar(counts.index,counts.values,color=WONG['skyblue'],edgecolor='k',lw=.5,alpha=.9)
for x,v in zip(counts.index,counts.values): ax[0].text(x,v*1.15,f'{v:,}',ha='center',fontsize=8.5)
ax[0].set_yscale('log'); ax[0].set_xlabel('Heavy atoms'); ax[0].set_ylabel('Enumerated molecules (log scale)'); ax[0].set_title('(a) Size of the enumerated space',fontsize=11); ax[0].set_ylim(1,1e5)
rec=[QM9_RECOVERY_PCT if 'QM9_RECOVERY_PCT' in dir() else float('nan'),100.0]
bars=ax[1].bar(['QM9 molecules\n(in range)','Confirmed interstellar\nspecies (in range)'],rec,color=[WONG['green'],WONG['orange']],edgecolor='k',lw=.5,alpha=.9,width=.55)
for b,v in zip(bars,rec): ax[1].text(b.get_x()+b.get_width()/2,v+.5,f'{v:.1f}%',ha='center',fontsize=11,fontweight='bold')
ax[1].set_ylabel('Recovered by the enumeration (%)'); ax[1].set_ylim(90,103); ax[1].set_title('(b) Recovery of reference molecules',fontsize=11); ax[1].axhline(100,ls='--',color='grey',lw=1,alpha=.6)
plt.tight_layout(); save('FigureS1_enumeration')
# S2 control experiments (charge separation / fluorine values from the corresponding control runs; calibration from M3)
fig,ax=plt.subplots(1,3,figsize=(13,3.9))
ax[0].plot(range(3),[0.660,0.525,0.437],'o-',color=WONG['vermillion'],lw=2.2,ms=11,mec='k',mew=.6); ax[0].axhline(.5,ls='--',color='grey',lw=1)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(['Neutral\nclosed-shell','+ charge-separated\n(≤ 3 heavy atoms)','+ charge-separated\n(all sizes)'],fontsize=8.5)
for i,v in enumerate([0.660,0.525,0.437]): ax[0].annotate(f'{v:.3f}',(i,v),textcoords="offset points",xytext=(0,10),ha='center',fontsize=9)
ax[0].set_ylabel('Same-size AUC of the dipole moment'); ax[0].set_ylim(.40,.70); ax[0].set_title('(a) Admitting charge-separated structures',fontsize=10.5)
bars=ax[1].bar(['Enumerated\nspace','Top-50\ncandidates','Confirmed\ninterstellar species'],[25,48,0],color=[WONG['orange'],WONG['orange'],WONG['green']],edgecolor='k',lw=.5,alpha=.9,width=.6)
for b,v in zip(bars,[25,48,0]): ax[1].text(b.get_x()+b.get_width()/2,v+1.2,f'{v}%',ha='center',fontsize=10,fontweight='bold')
ax[1].set_ylabel('Fluorine-containing (%)'); ax[1].set_ylim(0,56); ax[1].set_title('(b) Admitting fluorine (CHNOF space)',fontsize=10.5)
if CAL:
    raw=[RAW_MAE,v_mu.mean()]; cal=[DML_MAE,v_cal.mean()]; x=np.arange(2); w=.35
    b1=ax[2].bar(x-w/2,raw,w,label='GFN2-xTB',color=WONG['skyblue'],edgecolor='k',lw=.5); b2=ax[2].bar(x+w/2,cal,w,label='Δ-ML toward B3LYP/6-31G(2df,p)',color=WONG['purple'],edgecolor='k',lw=.5)
    ax[2].set_xticks(x); ax[2].set_xticklabels(['MAE vs B3LYP/6-31G(2df,p) (D)','Same-size AUC'],fontsize=8.5); ax[2].set_ylim(0,.8)
    for bb in (b1,b2):
        for b in bb: ax[2].text(b.get_x()+b.get_width()/2,b.get_height()+.015,f'{b.get_height():.3f}',ha='center',fontsize=8.5)
    ax[2].legend(fontsize=8,loc='upper right')
else: ax[2].text(.5,.5,'Δ-ML correction not run\n(QM9 reference file missing)',ha='center',transform=ax[2].transAxes); ax[2].axis('off')
ax[2].set_title('(c) Effect of the Δ-ML dipole correction',fontsize=10.5); plt.tight_layout(); save('FigureS2_controls')
# S3 landscape with discrete colours
fig,ax=plt.subplots(figsize=(8,5.4)); non=d2[d2.is_ism==0]; isd=d2[d2.is_ism==1]
for n in [6,5,4,3,2]:
    s=non[non.heavy_atoms==n]; ax.scatter(s.B,s.mu_xtb,s=8,alpha=.35,color=SIZECOL[n],edgecolors='none',label=f'{n} heavy atoms',rasterized=True)
ax.scatter(isd.B,isd.mu_xtb,marker='*',s=160,color=WONG['vermillion'],edgecolor='k',lw=.6,zorder=6,label='Confirmed interstellar species')
ax.set_xscale('log'); ax.set_xlabel('Rotational constant B (GHz, log scale)'); ax.set_ylabel('GFN2-xTB dipole moment μ (D)'); ax.legend(fontsize=8.5,loc='upper right',framealpha=.95,markerscale=2); ax.set_ylim(-.3,10)
plt.tight_layout(); save('FigureS3_landscape')
# S4 descriptor/fingerprint size correlations + GIN training curves
fig,ax=plt.subplots(1,3,figsize=(13,3.9)); dn=['MolLogP','TPSA','NumHDonors','NumHAcceptors','NumRotatableBonds','RingCount','FractionCSP3']
ax[0].barh(dn,[spearmanr(col,h2)[0] for col in SF2.T],color=WONG['green'],edgecolor='k',lw=.5); ax[0].barh(['Morgan fingerprint\n(number of set bits)'],[spearmanr(bits,h2)[0]],color=WONG['vermillion'],edgecolor='k',lw=.5)
ax[0].axvline(0,color='k',lw=.8); ax[0].set_xlim(-.6,.6); ax[0].set_xlabel('Spearman ρ with heavy-atom count'); ax[0].set_title('(a) The seven descriptors',fontsize=10.5)
bp=ax[1].boxplot([bits[h2==n] for n in [2,3,4,5,6] if (h2==n).any()],positions=[n for n in [2,3,4,5,6] if (h2==n).any()],widths=.6,patch_artist=True,showfliers=False)
for p_,n in zip(bp['boxes'],[n for n in [2,3,4,5,6] if (h2==n).any()]): p_.set_facecolor(SIZECOL[n]); p_.set_alpha(.8)
ax[1].set_xlabel('Heavy atoms'); ax[1].set_ylabel('Set bits in the 2048-bit Morgan fingerprint'); ax[1].set_title('(b) Set bits at each heavy-atom count',fontsize=10.5)
cur=np.array(gin_curves)
for c_ in cur: ax[2].plot(range(1,cur.shape[1]+1),c_,color=WONG['purple'],alpha=.25,lw=.8)
ax[2].plot(range(1,cur.shape[1]+1),cur.mean(0),color=WONG['blue'],lw=2,label=f'mean over {len(cur)} networks ({N_GIN_SEEDS} per fold)'); ax[2].set_yscale('log'); ax[2].set_xlabel('Epoch'); ax[2].set_ylabel('Weighted binary cross-entropy (training)'); ax[2].set_title('(c) GIN training loss, one curve per network',fontsize=10.5); ax[2].legend(fontsize=8.5)
plt.tight_layout(); save('FigureS4_descriptors_gin')
# S5 pairwise significance (readable colours, black text)
labels=['Size\ncontrol','Dipole μ\n(xTB)','μ²/Q$_{rot}$','Tanimoto','GIN','XGBoost']; n=len(labels); P=np.full((n,n),np.nan)
for i in range(n):
    for j in range(n):
        if i!=j: dd=vecs[i]-vecs[j]; P[i,j]=np.mean(np.abs([np.mean(dd*g.choice([-1,1],len(dd))) for _ in range(3000)])>=abs(dd.mean()))
from matplotlib.colors import ListedColormap
fig,ax=plt.subplots(figsize=(7.4,6.2)); im=ax.imshow(P,cmap=ListedColormap(plt.cm.YlGn_r(np.linspace(.15,.85,256))),vmin=0,vmax=.1,aspect='auto')
ax.set_xticks(range(n)); ax.set_yticks(range(n)); ax.set_xticklabels(labels,fontsize=8.5); ax.set_yticklabels(labels,fontsize=8.5); ax.grid(False)
for i in range(n):
    for j in range(n):
        if i!=j: ax.text(j,i,'< 0.001' if P[i,j]<0.001 else f'{P[i,j]:.3f}',ha='center',va='center',fontsize=8,color='black')
plt.colorbar(im,ax=ax).set_label('Paired permutation p value (capped at 0.1)'); ax.set_xlabel('Column predictor'); ax.set_ylabel('Row predictor'); plt.tight_layout(); save('FigureS5_pairwise')
# Graphical abstract (two panels; no rotational constant; no continuous colour map)
fig,ax=plt.subplots(1,2,figsize=(12.5,4.8))
ax[0].bar([x-.2 for x in sorted(qm9)],[qm9[x] for x in sorted(qm9)],width=.4,color=WONG['vermillion'],alpha=.85,edgecolor='white',lw=.5,label='QM9')
ax[0].bar([x+.2 for x in sorted(astro)],[astro[x] for x in sorted(astro)],width=.4,color=WONG['skyblue'],alpha=.85,edgecolor='white',lw=.5,label='AstroSpace')
for x in sorted(ism): ax[0].scatter(x,ism[x],marker='*',s=190,color=WONG['orange'],edgecolor='#5a3000',lw=.8,zorder=6)
ax[0].scatter([],[],marker='*',s=150,color=WONG['orange'],edgecolor='#5a3000',lw=.8,label='Confirmed interstellar species'); ax[0].set_yscale('log'); ax[0].set_xlabel('Number of heavy atoms',fontsize=12); ax[0].set_ylabel('Number of molecules (log scale)',fontsize=12)
ax[0].set_title('QM9 is dominated by nine-heavy-atom molecules;\nconfirmed interstellar molecules are small',fontsize=11.5); ax[0].legend(fontsize=9.5,loc='upper left',framealpha=.95); ax[0].set_xticks(range(1,10)); ax[0].set_ylim(.7,2e5)
ga_n=['Size-only\ncontrol','Dipole\nmoment','Tanimoto\nsimilarity','Graph neural\nnetwork','Descriptor\nXGBoost']; ga_v=[v_ctrl.mean(),v_mu.mean(),v_tan.mean(),v_gin.mean(),v_xgb.mean()]
bars=ax[1].bar(ga_n,ga_v,color=[GREY,WONG['green'],WONG['skyblue'],WONG['purple'],WONG['orange']],alpha=.9,edgecolor='#333',lw=.6,width=.62)
ax[1].axhline(.5,ls='--',color=WONG['vermillion'],lw=1.3,alpha=.8); ax[1].text(4.4,.512,'random',fontsize=9,color=WONG['vermillion'],ha='right',style='italic')
for b,v in zip(bars,ga_v): ax[1].text(b.get_x()+b.get_width()/2,v+.012,f'{v:.2f}',ha='center',fontsize=10,fontweight='bold')
ax[1].set_ylim(.42,.99); ax[1].set_ylabel('Same-size AUC on AstroSpace (0.5 = random)',fontsize=11.5); ax[1].set_title('After controlling for size, a zero-parameter physical property\nis significant, and a graph network and seven simple descriptors perform equally',fontsize=11.5)
plt.tight_layout(); plt.savefig(FIG/'Graphical_abstract.png',dpi=300,facecolor='white',bbox_inches='tight',pad_inches=.2); plt.savefig(FIG/'Graphical_abstract.pdf',facecolor='white',bbox_inches='tight',pad_inches=.2); plt.show()
print("Supplementary figures S1-S5 and graphical abstract written to",FIG)


In [ ]:
# ============================================================
# CELL 10 — Export the complete project
# ============================================================
import shutil, json

# 1. frozen numbers for the manuscript
summary = {
 "space": {"max_heavy": MAX_HEAVY, "enumerated": int(len(space)),
           "xtb_converged": int(len(phys)), "ism_positives": int(phys.is_ism.sum()),
           "evaluable_positives": int(len(v_mu)), "prevalence_pct": float(phys.is_ism.mean()*100)},
 "size_matched_AUC": {
     "size_control": float(v_ctrl.mean()), "dipole_raw": float(v_mu.mean()),
     "detectability": float(v_D.mean()), "tanimoto": float(v_tan.mean()),
     "xgboost_size_free": float(v_xgb.mean())},
 "charge_separation": bool(ALLOW_CHARGE_SEPARATION),
 "include_fluorine": bool(INCLUDE_FLUORINE),
 "calibrated": bool(CALIBRATED),
}
json.dump(summary, open(OUT/"astrospace_summary.json","w"), indent=2)
print(json.dumps(summary, indent=2))


# ---- every number quoted in the paper, in one file ----
_ci=lambda v: [float(q) for q in np.percentile([np.mean(g.choice(v,len(v),replace=True)) for _ in range(10000)],[2.5,97.5])]
_p=lambda v: float(np.mean(np.abs([np.mean((v-0.5)*g.choice([-1,1],len(v))) for _ in range(10000)])>=abs(v.mean()-0.5)))
_pair=lambda a,b: (float((a-b).mean()), float(np.mean(np.abs([np.mean((a-b)*g.choice([-1,1],len(a))) for _ in range(10000)])>=abs((a-b).mean()))))
paper={"space":{"enumerated":int(len(space)),"converged":int(len(phys)),"analysed_2to6":int(len(d2)),"detected":int(len(pos2)),"undetected":int(len(unl2))},
 "table2":{k:{"auc":float(v.mean()),"ci":_ci(v),"p":_p(v)} for k,v in {"size_control":v_ctrl,"dipole_xtb":v_mu,"mu2_over_Qrot":v_D,"tanimoto":v_tan,"gin":v_gin,"xgboost":v_xgb}.items()},
 "table1_counterfactual":CF,
 "pairwise":{"dipole_minus_size":_pair(v_mu,v_ctrl),"mu2Q_minus_dipole":_pair(v_D,v_mu),"xgb_minus_tan":_pair(v_xgb,v_tan),"gin_minus_tan":_pair(v_gin,v_tan),"gin_minus_xgb":_pair(v_gin,v_xgb)},
 "candidates_top50_size_counts":{int(k):int(v) for k,v in ranked.head(50).heavy_atoms.value_counts().items()}}
if CAL: paper["delta_ml"]={"n_pairs":int(len(ref_pairs)),"raw_mae":RAW_MAE,"dml_mae":DML_MAE,"same_size_auc":float(v_cal.mean()),"ci":_ci(v_cal),"p":_p(v_cal)}
paper["positives"]={"census_in_range":int(N_ISM_NEUTRAL+N_ISM_CS),"charge_separated":int(N_ISM_CS),"in_space":int(space.is_ism.sum()),
    "converged":int(phys.is_ism.sum()),"analysed_2to6":int(len(pos2)),"per_heavy_atom_converged":{int(k):int(v) for k,v in phys[phys.is_ism==1].groupby("heavy_atoms").size().items()},
    "not_converged":[str(s) for s in space[(space.is_ism==1)&(~space.smiles.isin(set(phys.smiles)))].smiles],
    "zero_dipole_analysed":int((d2[(d2.is_ism==1)].mu_xtb<0.05).sum())}
paper["prospective"]=tS3.to_dict("records")
paper["size_only_whole_dataset_auc"]={"qm9":QM9_SIZE_ONLY_AUC,"astrospace":ASTRO_SIZE_ONLY_AUC}
paper["strata"]={int(k):{"molecules":int(v),"undetected":int(u)} for (k,v),u in zip(d2.groupby("heavy_atoms").size().items(), d2[d2.is_ism==0].groupby("heavy_atoms").size().values)}
json.dump(paper,open(OUT/"paper_numbers.json","w"),indent=2); print("paper_numbers.json written")

# 2. bundle everything: data, figures, candidate list, summaries
bundle = BASE/"AstroSpace_project"
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
# copy result CSVs and JSONs, but skip the xTB checkpoint: it is a mid-run
# safety copy that duplicates AstroSpace6_xtb.csv exactly, so bundling it would
# double the data. The final AstroSpace6_xtb.csv already contains everything.
for f in OUT.glob("*.csv"):
    if f.name == "xtb_checkpoint.csv":
        continue
    shutil.copy(f, bundle)
for f in OUT.glob("*.json"): shutil.copy(f, bundle)
figdir = bundle/"figures"; figdir.mkdir()
for f in FIG.glob("*"): shutil.copy(f, figdir)

shutil.make_archive(str(BASE/"AstroSpace_project"), "zip", bundle)
print(f"\\nComplete project archived -> {BASE/'AstroSpace_project.zip'}")
print("Contents: enumerated space, GFN2-xTB properties, candidate tables, Figures 1-5, S1-S5 and the graphical abstract (PNG+PDF), summaries.")
if ON_KAGGLE:
    print("On Kaggle, everything under /kaggle/working is saved as the notebook output.")


---
## Outputs

`AstroSpace_project/` contains the enumerated space (`AstroSpace6_enumerated.csv`), the computed
properties (`AstroSpace6_xtb.csv`), the candidate tables (`TableS1_top50_candidates.csv`,
`TableS2_top5_per_stratum.csv`, `top50_novel_candidates.csv`), the figures (300 dpi PNG and PDF) and
the summaries (`paper_numbers.json`, `size_matched_results.json`, `gin_and_counterfactual.json`,
`dipole_calibration.json`, `astrospace_summary.json`). Every value quoted in the manuscript is in
`paper_numbers.json`.
